# Patient Readmission Classification

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `readmitted`

## 2 · Project Overview

We predict **30-day hospital readmission** based on patient demographics, hospitalization complexity (diagnoses, procedures, medications), discharge disposition, prior admissions, and comorbidities.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a patient's age, length of stay, diagnoses, procedures, medications, lab work, prior admissions, discharge type, diabetes status, and admission type, predict whether they will be readmitted within 30 days.

## 5 · Why This Project Matters

- **Hospital readmissions cost US healthcare ~$26 billion annually.**
- CMS penalizes hospitals with excessive 30-day readmission rates.
- Early identification enables **targeted post-discharge interventions**.
- Demonstrates healthcare classification with operational and financial impact.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | age, length_of_stay, num_diagnoses, num_procedures, num_medications, num_lab_procedures, num_prior_admissions, discharge_disposition, has_diabetes, emergency_admission |
| **Target** | readmitted (0 = no, 1 = yes within 30 days) |
| **Class balance** | ~70/30 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "readmitted"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: readmitted
Save dir: E:\Github\Machine-Learning-Projects\Classification\Patient Readmission Classification


## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 hospital discharge records with 30-day readmission outcome.

In [4]:
np.random.seed(SEED)
n = 8000
age = np.random.randint(18, 95, n)
length_of_stay = np.random.poisson(5, n).clip(1, 30)
num_diagnoses = np.random.poisson(4, n).clip(1, 15)
num_procedures = np.random.poisson(1.5, n).clip(0, 10)
num_medications = np.random.poisson(8, n).clip(1, 25)
num_lab_procedures = np.random.poisson(40, n).clip(5, 100)
num_prior_admissions = np.random.poisson(1, n).clip(0, 8)
discharge_disposition = np.random.choice(["Home", "Facility", "AMA"], n, p=[0.6, 0.3, 0.1])
disp_num = np.where(discharge_disposition == "Home", 0,
                    np.where(discharge_disposition == "Facility", 1, 2))
has_diabetes = np.random.binomial(1, 0.3, n)
emergency_admission = np.random.binomial(1, 0.4, n)

score = (0.01 * age + 0.03 * length_of_stay + 0.06 * num_diagnoses
         + 0.05 * num_procedures + 0.02 * num_medications
         + 0.15 * num_prior_admissions + 0.3 * disp_num
         + 0.2 * has_diabetes + 0.15 * emergency_admission
         + np.random.normal(0, 1.0, n))
readmitted = (score > np.percentile(score, 70)).astype(int)

df = pd.DataFrame({"age": age, "length_of_stay": length_of_stay,
                    "num_diagnoses": num_diagnoses, "num_procedures": num_procedures,
                    "num_medications": num_medications, "num_lab_procedures": num_lab_procedures,
                    "num_prior_admissions": num_prior_admissions,
                    "discharge_disposition": discharge_disposition,
                    "has_diabetes": has_diabetes, "emergency_admission": emergency_admission,
                    "readmitted": readmitted})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['readmitted'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (8000, 11)
Class balance:
readmitted
0    0.7
1    0.3
Name: proportion, dtype: float64


,age,length_of_stay,num_diagnoses,num_procedures,num_medications,num_lab_procedures,num_prior_admissions,discharge_disposition,has_diabetes,emergency_admission,readmitted
0,69,5,5,0,7,40,1,Home,0,1,0
1,32,4,3,0,15,37,2,Home,1,0,0
2,89,7,5,0,8,42,1,Home,0,0,1
3,78,3,5,1,8,35,2,Facility,0,0,1
4,38,3,3,3,9,29,2,Home,1,1,0


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (8000, 11)

Missing values:
age                      0
length_of_stay           0
num_diagnoses            0
num_procedures           0
num_medications          0
num_lab_procedures       0
num_prior_admissions     0
discharge_disposition    0
has_diabetes             0
emergency_admission      0
readmitted               0
dtype: int64

Duplicate rows: 0

Dtypes:
age                       int32
length_of_stay            int32
num_diagnoses             int32
num_procedures            int32
num_medications           int32
num_lab_procedures        int32
num_prior_admissions      int32
discharge_disposition    object
has_diabetes              int32
emergency_admission       int32
readmitted                int64
dtype: object

Target 'readmitted' confirmed. Value counts:
readmitted
0    5600
1    2400
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = ["age", "length_of_stay", "num_diagnoses", "num_medications", "num_prior_admissions", "num_lab_procedures"]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3][i % 3]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Readmission Status", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.groupby("discharge_disposition")[TARGET].mean().plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Readmission Rate by Discharge Disposition")
df.groupby("emergency_admission")[TARGET].mean().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Readmission Rate by Emergency Admission")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `readmitted`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "salmon"], edgecolor="black")
ax.set_title("30-Day Readmission Distribution")
ax.set_xticklabels(["Not Readmitted (0)", "Readmitted (1)"], rotation=0)
plt.tight_layout()
plt.show()
print(f"Readmission rate: {df[TARGET].mean():.1%}")

Readmission rate: 30.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (6400, 10), Test: (1600, 10)
Train class distribution:
readmitted
0    0.7
1    0.3
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `discharge_disposition` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: ~30% readmitted.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["complexity_score"] = X_train["num_diagnoses"] + X_train["num_procedures"] + X_train["num_medications"] * 0.5
X_test["complexity_score"] = X_test["num_diagnoses"] + X_test["num_procedures"] + X_test["num_medications"] * 0.5

X_train["care_intensity"] = X_train["num_lab_procedures"] / (X_train["length_of_stay"] + 1)
X_test["care_intensity"] = X_test["num_lab_procedures"] / (X_test["length_of_stay"] + 1)

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (12): ['age', 'length_of_stay', 'num_diagnoses', 'num_procedures', 'num_medications', 'num_lab_procedures', 'num_prior_admissions', 'discharge_disposition', 'has_diabetes', 'emergency_admission', 'complexity_score', 'care_intensity']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.7000
F1       : 0.2357

              precision    recall  f1-score   support

           0       0.72      0.93      0.81      1120
           1       0.50      0.15      0.24       480

    accuracy                           0.70      1600
   macro avg       0.61      0.54      0.52      1600
weighted avg       0.65      0.70      0.64      1600



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                       
NearestCentroid             0.605625           0.596280  0.637420  0.620940   0.658067  0.605625    0.034085
GaussianNB                  0.688750           0.556845  0.625086  0.651003   0.648163  0.688750    0.027822
AdaBoostClassifier          0.703750           0.554464  0.636765  0.650540   0.663248  0.703750    0.349730
LogisticRegression          0.700625           0.545089  0.638891  0.641126   0.655624  0.700625    0.034532
LinearDiscriminantAnalysis  0.697500           0.544048  0.638633  0.640173   0.650571  0.697500    0.034747
RidgeClassifier             0.704375           0.541220  0.638640  0.636498   0.661788  0.704375    0.020605
LinearSVC                   0.703125           0.540327  0.638627  0.635649   0.658941  0.7031

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.6281
F1: 0.3025


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.1643  (1.2s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[30]	valid_0's binary_logloss: 0.58826
LightGBM F1: 0.1013  (0.7s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  0.6281  0.3025     0.3458  0.2687
Logistic Regression    0.7000  0.2357     0.5000  0.1542
CatBoost               0.7075  0.1643     0.5750  0.0958
LightGBM               0.7006  0.1013     0.5094  0.0563

Top 3 models: ['FLAML', 'Logistic Regression', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  FLAML:
    Accuracy : 0.6281
    F1       : 0.3025
    Precision: 0.3458
    Recall   : 0.2687

  Logistic Regression:
    Accuracy : 0.7000
    F1       : 0.2357
    Precision: 0.5000
    Recall   : 0.1542

  CatBoost:
    Accuracy : 0.7075
    F1       : 0.1643
    Precision: 0.5750
    Recall   : 0.0958


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: FLAML

Classification Report:


              precision    recall  f1-score   support

           0       0.71      0.78      0.75      1120
           1       0.35      0.27      0.30       480

    accuracy                           0.63      1600
   macro avg       0.53      0.53      0.52      1600
weighted avg       0.60      0.63      0.61      1600


Total errors: 595 / 1600 (37.2%)

Sample misclassifications:
      age  length_of_stay  num_diagnoses  num_procedures  num_medications  num_lab_procedures  num_prior_admissions  discharge_disposition  has_diabetes  emergency_admission  complexity_score  care_intensity  actual  predicted  correct
2056   20               1              4               1                7                  42                     0                    1.0             0                    0               8.5       21.000000       1          0    False
3965   90               6              5               2                9                  34                     1                    1.0 

## 25 · Interpretation and Business Insight

**Key findings:**
- **Prior admissions** are the strongest readmission predictor (recidivism).
- **Discharge to facility/AMA** substantially increases readmission risk.
- **Diabetes** and **emergency admission** add significant risk.
- **Case complexity** (diagnoses + procedures + medications) correlates with readmission.

**Business takeaway:** Focus post-discharge programs on patients with multiple prior admissions, complex cases discharged AMA, and diabetic emergency patients.

## 26 · Limitations

1. Synthetic data — real readmission has complex social determinants.
2. No social determinants of health (housing, income, support).
3. Missing medication adherence data.
4. No follow-up appointment data.
5. Binary 30-day window ignores timing.

## 27 · How to Improve This Project

1. Use real hospital data (MIMIC-IV, Diabetes 130-US Hospitals).
2. Add social determinants of health.
3. Include medication adherence and follow-up scheduling.
4. Model time-to-readmission (survival analysis).
5. Add post-discharge care plan features.

## 28 · Production Considerations

- Deploy as a discharge risk scoring tool.
- Trigger care coordination for high-risk patients.
- Output risk probability for clinical prioritization.
- Monitor readmission rates by department.
- Integrate with care transition programs.

## 29 · Common Mistakes

1. Using accuracy on imbalanced readmission data.
2. Not considering social determinants.
3. Ignoring the cost asymmetry (missed readmission vs. extra follow-up).
4. Treating readmission as purely a clinical problem.
5. Not validating across different patient populations.

## 30 · Mini Challenge / Exercises

1. Remove `num_prior_admissions` — how much does recall drop?
2. Set threshold for 80% recall — what precision is achieved?
3. Analyze readmission rate by age group deciles.
4. Compare models for diabetic vs. non-diabetic subgroups.
5. Calculate the cost-benefit of intervening on top 20% highest risk.

## 31 · Final Summary / Key Takeaways

1. **Prior admissions** are the strongest readmission predictor.
2. **Discharge disposition** (AMA, facility) signals risk.
3. **Case complexity** (diagnoses + procedures) correlates with risk.
4. **Diabetes** and **emergency admission** are key comorbidity flags.
5. **Cost-sensitive thresholds** matter — missed readmissions are expensive.
6. **Social determinants** (missing here) are critical in real-world models.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Patient Readmission Classification\metrics.json
{
  "CatBoost": {
    "accuracy": 0.7075,
    "f1": 0.1643,
    "precision": 0.575,
    "recall": 0.0958
  },
  "LightGBM": {
    "accuracy": 0.7006,
    "f1": 0.1013,
    "precision": 0.5094,
    "recall": 0.0563
  },
  "Logistic Regression": {
    "accuracy": 0.7,
    "f1": 0.2357,
    "precision": 0.5,
    "recall": 0.1542
  },
  "FLAML": {
    "accuracy": 0.6281,
    "f1": 0.3025,
    "precision": 0.3458,
    "recall": 0.2687
  }
}
